# Homework 11 - Domain Adaptation (DaNN)

以 baseline 為基礎，加入以下改進：
- **λ 動態調整**：使用 DANN 論文標準 schedule
- **更多訓練 epoch**：500 epoch
- **強化 Data Augmentation**：RandomCrop
- **Pseudo Labeling**：每 50 epoch 更新一次 pseudo label，利用 test data label-balanced 特性

In [ ]:
# 下載並解壓資料集
!gdown --id '1P4fGNb9JhJj8W0DA_Qrp7mbrRHfF5U_f' --output real_or_drawing.zip
!unzip real_or_drawing.zip

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset, ConcatDataset
import cv2
import pandas as pd
from PIL import Image
import os

# 確認 GPU 可用
print(f'使用裝置: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 資料前處理

- **Source（真實照片）**：先轉灰階，再用 Canny Edge Detection 抽出輪廓，使其視覺上接近手繪圖
- **Target（手繪圖）**：轉灰階後 resize 到 32×32
- 新增 `RandomCrop(32, padding=4)` 以增加資料多樣性

In [ ]:
# Source domain transform：真實照片 → Canny 邊緣 → 近似手繪風格
source_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Lambda(lambda x: cv2.Canny(np.array(x), 170, 300)),
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15, fill=(0,)),
    transforms.RandomCrop(32, padding=4, fill=0),  # 新增：隨機裁切增加多樣性
    transforms.ToTensor(),
])

# Target domain transform：訓練時使用（帶 augmentation）
target_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15, fill=(0,)),
    transforms.RandomCrop(32, padding=4, fill=0),  # 新增
    transforms.ToTensor(),
])

# Test transform：推論時不做 augmentation，確保結果穩定
test_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

# 建立 Dataset
source_dataset = ImageFolder('real_or_drawing/train_data', transform=source_transform)
target_dataset = ImageFolder('real_or_drawing/test_data', transform=target_transform)
test_dataset   = ImageFolder('real_or_drawing/test_data', transform=test_transform)

# 建立 DataLoader
source_dataloader = DataLoader(source_dataset, batch_size=32, shuffle=True,  num_workers=2)
target_dataloader = DataLoader(target_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_dataloader   = DataLoader(test_dataset,   batch_size=128, shuffle=False, num_workers=2)

print(f'Source 資料數: {len(source_dataset)}')
print(f'Target 資料數: {len(target_dataset)}')

## 模型架構

DaNN 由三個子模型組成：
- **FeatureExtractor**：CNN，將 32×32 圖片壓縮為 512 維 feature vector
- **LabelPredictor**：全連接層，從 feature 預測 10 類別
- **DomainClassifier**：全連接層，判斷 feature 來自 source 還是 target domain

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 → 16

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 16 → 8

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 8 → 4

            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 4 → 2

            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 2 → 1
        )

    def forward(self, x):
        # 使用 view 取代 squeeze，避免 batch_size=1 時維度消失的 bug
        x = self.conv(x)
        return x.view(x.size(0), -1)


class LabelPredictor(nn.Module):
    def __init__(self):
        super(LabelPredictor, self).__init__()
        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, h):
        return self.layer(h)


class DomainClassifier(nn.Module):
    def __init__(self):
        super(DomainClassifier, self).__init__()
        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

    def forward(self, h):
        return self.layer(h)

In [ ]:
# 初始化模型與 Optimizer
feature_extractor = FeatureExtractor().cuda()
label_predictor   = LabelPredictor().cuda()
domain_classifier = DomainClassifier().cuda()

class_criterion  = nn.CrossEntropyLoss()
domain_criterion = nn.BCEWithLogitsLoss()

optimizer_F = optim.Adam(feature_extractor.parameters())
optimizer_C = optim.Adam(label_predictor.parameters())
optimizer_D = optim.Adam(domain_classifier.parameters())

## λ 動態調整（Medium Baseline 關鍵）

原始 baseline 使用固定 λ=0.1。
DANN 論文建議：訓練初期讓 classifier 先學好分類，再逐漸加強 domain alignment。

公式：`λ = 2 / (1 + exp(-10·p)) - 1`，p 從 0 增長到 1

In [ ]:
def get_lambda(epoch, max_epoch):
    """DANN 論文的 lambda schedule，從 0 平滑增長到 ~1"""
    p = epoch / max_epoch
    return 2 / (1 + np.exp(-10 * p)) - 1

## Pseudo Labeling（Strong Baseline 關鍵）

HW11 PDF 提示：**Test data is label-balanced**（10 類各 10000 張）。

做法：
1. 對 100000 張 target 圖片做推論，取得每張的類別預測與信心值
2. 對每個類別，取信心值最高的前 N 張作為 pseudo label
3. 將這些 pseudo-labeled 資料加入訓練集，視為有標籤的資料

In [ ]:
class PseudoDataset(Dataset):
    """存放 pseudo-labeled 的 target domain 圖片"""
    def __init__(self, img_paths, labels, transform):
        self.img_paths = img_paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img   = Image.open(self.img_paths[idx])
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


def generate_pseudo_labels(feature_extractor, label_predictor, test_dataset, n_per_class=300):
    """
    對 test_dataset 做推論，每個類別取信心值最高的 n_per_class 張，
    回傳 PseudoDataset 供訓練使用。
    """
    feature_extractor.eval()
    label_predictor.eval()

    all_probs = []
    loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

    with torch.no_grad():
        for imgs, _ in loader:
            imgs  = imgs.cuda()
            feat  = feature_extractor(imgs)
            probs = F.softmax(label_predictor(feat), dim=1)
            all_probs.append(probs.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)  # [100000, 10]

    # 取得所有圖片路徑（ImageFolder 的 samples 是 (path, class_idx) 的 list）
    all_paths = [path for path, _ in test_dataset.samples]

    pseudo_paths, pseudo_labels = [], []
    for cls in range(10):
        # 取該類別信心值最高的前 n_per_class 張
        cls_conf   = all_probs[:, cls]
        top_idx    = np.argsort(cls_conf)[::-1][:n_per_class]
        for idx in top_idx:
            pseudo_paths.append(all_paths[idx])
            pseudo_labels.append(cls)

    feature_extractor.train()
    label_predictor.train()

    print(f'  Pseudo label 更新完成，共 {len(pseudo_paths)} 筆（每類 {n_per_class} 筆）')
    return PseudoDataset(pseudo_paths, pseudo_labels, target_transform)

## 訓練函式

DaNN 每個 iteration 分兩步：
1. **訓練 Domain Classifier**：讓它能分辨 source/target（固定 feature extractor）
2. **訓練 Feature Extractor + Label Predictor**：
   - 最小化 classification loss（讓預測準確）
   - 最大化 domain loss（騙過 domain classifier，使 feature 看起來 domain-invariant）
   - 整體 Loss = classification_loss - λ × domain_loss

In [ ]:
def train_epoch(source_dataloader, target_dataloader, lamb):
    running_D_loss, running_F_loss = 0.0, 0.0
    total_hit, total_num = 0.0, 0.0

    for i, ((source_data, source_label), (target_data, _)) in enumerate(
        zip(source_dataloader, target_dataloader)
    ):
        source_data  = source_data.cuda()
        source_label = source_label.cuda()
        target_data  = target_data.cuda()

        # 混合 source + target，domain_label: source=1, target=0
        mixed_data   = torch.cat([source_data, target_data], dim=0)
        domain_label = torch.zeros([source_data.shape[0] + target_data.shape[0], 1]).cuda()
        domain_label[:source_data.shape[0]] = 1

        feature = feature_extractor(mixed_data)

        # Step 1：訓練 Domain Classifier（feature detach，不更新 extractor）
        domain_logits = domain_classifier(feature.detach())
        loss_D = domain_criterion(domain_logits, domain_label)
        running_D_loss += loss_D.item()
        loss_D.backward()
        optimizer_D.step()

        # Step 2：訓練 Feature Extractor + Label Predictor
        class_logits  = label_predictor(feature[:source_data.shape[0]])
        domain_logits = domain_classifier(feature)
        # 分類 loss 最小化 + domain loss 最大化（加負號）
        loss_F = class_criterion(class_logits, source_label) \
                 - lamb * domain_criterion(domain_logits, domain_label)
        running_F_loss += loss_F.item()
        loss_F.backward()
        optimizer_F.step()
        optimizer_C.step()

        optimizer_D.zero_grad()
        optimizer_F.zero_grad()
        optimizer_C.zero_grad()

        total_hit += torch.sum(torch.argmax(class_logits, dim=1) == source_label).item()
        total_num += source_data.shape[0]
        print(i, end='\r')

    return running_D_loss / (i + 1), running_F_loss / (i + 1), total_hit / total_num

## 主訓練迴圈

- 訓練 500 epoch
- 每 50 epoch 更新一次 pseudo label（warm-up 50 epoch 後才開始）
- λ 依 schedule 動態調整

In [ ]:
MAX_EPOCH      = 500
PSEUDO_START   = 50   # 從第幾 epoch 開始加入 pseudo label
PSEUDO_INTERVAL = 50  # 每幾個 epoch 更新一次 pseudo label
N_PER_CLASS    = 300  # 每類取幾張 pseudo-labeled 圖片

# 訓練開始時先用原始 source dataloader
current_source_dataloader = source_dataloader

for epoch in range(MAX_EPOCH):
    lamb = get_lambda(epoch, MAX_EPOCH)

    # Pseudo label 更新
    if epoch >= PSEUDO_START and (epoch - PSEUDO_START) % PSEUDO_INTERVAL == 0:
        print(f'[Epoch {epoch}] 更新 Pseudo Labels...')
        pseudo_dataset = generate_pseudo_labels(
            feature_extractor, label_predictor, test_dataset, n_per_class=N_PER_CLASS
        )
        # 將 pseudo-labeled target 資料與原始 source 資料合併
        combined = ConcatDataset([source_dataset, pseudo_dataset])
        current_source_dataloader = DataLoader(
            combined, batch_size=32, shuffle=True, num_workers=2
        )
        print(f'  合併後訓練集大小: {len(combined)}')

    train_D_loss, train_F_loss, train_acc = train_epoch(
        current_source_dataloader, target_dataloader, lamb
    )

    # 儲存模型
    torch.save(feature_extractor.state_dict(), 'extractor_model.bin')
    torch.save(label_predictor.state_dict(),   'predictor_model.bin')

    print(
        f'epoch {epoch:>3d}: '
        f'D loss={train_D_loss:.4f}, '
        f'F loss={train_F_loss:.4f}, '
        f'acc={train_acc:.4f}, '
        f'λ={lamb:.4f}'
    )

## 推論與輸出

推論時不做 augmentation，直接對全部 100000 張測試圖片預測類別。

In [ ]:
result = []
feature_extractor.eval()
label_predictor.eval()

with torch.no_grad():
    for test_data, _ in test_dataloader:
        test_data    = test_data.cuda()
        class_logits = label_predictor(feature_extractor(test_data))
        preds        = torch.argmax(class_logits, dim=1).cpu().numpy()
        result.append(preds)

result = np.concatenate(result)

df = pd.DataFrame({'id': np.arange(0, len(result)), 'label': result})
df.to_csv('DaNN_submission.csv', index=False)
print(f'完成！共 {len(result)} 筆預測，已存至 DaNN_submission.csv')
print(df.head(10))